In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix

In [3]:
df=pd.read_csv(r'C:\Users\adnan\OneDrive\Desktop\5TH SEM OJT\project 1_diabaties\data\processed\mohammed_adnan_p1w2_features.csv')

In [4]:
df.head()

,patient_id,age,gender,pregnancies,glucose,blood_pressure,skin_thickness,insulin,bmi,diabetes_pedigree,...,smoker,last_checkup_date,outcome,age_band,bmi_band,glucose_band,insulin_glucose,pregnancies_rate,since_checkup,activity_smoker
0,P00826,79,male,0,124.0,71.0,30.000000,134.67525,23.4,0.588,...,NO,2023-04-20,0,Old,Normal,Prediabetes,1.086091,0.000000,618,NaN
1,P00108,37,male,0,153.0,85.0,24.000000,42.00000,33.9,0.192,...,YES,2024-05-03,1,Middle-Aged,Obese,Diabetes,0.274510,0.000000,239,Medium/YES
2,P00517,39,female,0,142.0,68.0,22.000000,159.00000,26.9,0.777,...,NaN,2023-07-08,0,Middle-Aged,Overweight,Diabetes,1.119718,0.000000,539,NaN
3,P00687,68,female,7,121.0,88.0,26.406623,134.67525,29.0,1.217,...,YES,2024-04-14,1,Old,Overweight,Prediabetes,1.113019,0.102941,258,NaN
4,P00927,75,female,0,107.0,84.0,31.000000,101.00000,21.2,0.278,...,NO,2024-01-27,0,Old,Normal,Prediabetes,0.943925,0.000000,336,Low/NO


In [5]:
feature_cols=['age','pregnancies','glucose','blood_pressure','skin_thickness','insulin','bmi','diabetes_pedigree','bmi_band','glucose_band']



In [6]:
x=df[feature_cols]
y=df['outcome']

In [7]:
                                                               
x=pd.get_dummies(x,drop_first=True)

In [8]:
x_train,x_test,y_train,y_test= train_test_split(
x,y,test_size=0.2,random_state=42,stratify=y)

In [9]:
scaler = StandardScaler()
x_train_scaled=scaler.fit_transform(x_train)
x_test_scaled=scaler.transform(x_test)

In [10]:
logreg = LogisticRegression(max_iter=1000).fit(x_train_scaled, y_train)
dec_tree=DecisionTreeClassifier(max_depth=3).fit(x_train,y_train)
knn_best = KNeighborsClassifier(n_neighbors=25).fit(x_train_scaled, y_train)

In [11]:
print("Logistic Regression Accuracy:", accuracy_score(y_test, logreg.predict(x_test_scaled)))
print("Decision tree accuracy:",accuracy_score(y_test,dec_tree.predict(x_test)))
print("KNN accuracy:",accuracy_score(y_test,knn_best.predict(x_test_scaled)))


Logistic Regression Accuracy: 0.7578947368421053
Decision tree accuracy: 0.7210526315789474
KNN accuracy: 0.7368421052631579


In [12]:
print('test set:',len(y_test),'patients |',int(np.sum(y_test)),'of them diabetic')


test set: 190 patients | 55 of them diabetic


In [13]:
models = [("Logistic Regression", logreg, x_test_scaled),
          ("Decision Tree (d=3)",dec_tree, x_test),
        ("KNN (k-25)",  knn_best, x_test_scaled)]


build a confusion matrix for evermodel

In [14]:
rows = []
for name,m,Xt in models:
    cm = confusion_matrix(y_test,m.predict(Xt))
    tn, fp, fn, tp = cm.ravel()
    rows.append({"models": name, "correct_negative": int(tn),"False_alarms":int(fp),
                 "missed_patients":int(fn),"found_patients":int(tp),
                 "accuracy":accuracy_score(y_test,m.predict(Xt))})
print(rows)
matrixs=pd.DataFrame(rows)
matrixs

[{'models': 'Logistic Regression', 'correct_negative': 127, 'False_alarms': 8, 'missed_patients': 38, 'found_patients': 17, 'accuracy': 0.7578947368421053}, {'models': 'Decision Tree (d=3)', 'correct_negative': 121, 'False_alarms': 14, 'missed_patients': 39, 'found_patients': 16, 'accuracy': 0.7210526315789474}, {'models': 'KNN (k-25)', 'correct_negative': 129, 'False_alarms': 6, 'missed_patients': 44, 'found_patients': 11, 'accuracy': 0.7368421052631579}]


,models,correct_negative,False_alarms,missed_patients,found_patients,accuracy
0,Logistic Regression,127,8,38,17,0.757895
1,Decision Tree (d=3),121,14,39,16,0.721053
2,KNN (k-25),129,6,44,11,0.736842


                                  logistic regression                                                                                   

* **127 (TN):** 127 people were correctly identified as not having diabetes.
* **8 (FP):** 8 people were incorrectly identified as having diabetes.
* **39 (FN):** 39 people who actually had diabetes were missed by the model.
* **16 (TP):** 16 people who had diabetes were correctly identified by the model.

* **Logistic Regression:** The main error is **missed patients (FN = 39)**.
* **Decision Tree:** The main error is **missed patients (FN = 39)**, along with 14 false alarms.
* **KNN:** The main error is **missed patients (FN = 45)**, which is the highest among the three models.

score the model that never says yes

In [15]:
from sklearn.dummy import DummyClassifier

In [16]:
dummy=DummyClassifier(strategy='most_frequent')
dummy.fit(x_train_scaled,y_train)
y_pred=dummy.predict(x_test_scaled)
confusion_matrix(y_test,y_pred)

array([[135,   0],
       [ 55,   0]])

In [17]:
print("Dummy Classifier Accuracy:", accuracy_score(y_test, y_pred))

Dummy Classifier Accuracy: 0.7105263157894737


In [18]:
y_test.value_counts(normalize=True)             

outcome
0    0.710526
1    0.289474
Name: proportion, dtype: float64